Configure the project and run folders in the code before running. Supply your own image data and saved checkpoints.

# Matched 40% augmentation control

Use Colab with a GPU and run both code cells. The first cell checks the original split and the saved 40% run for each seed. The second trains the three matched controls and saves checkpoints and paired comparisons to a new Google Drive folder.

Each replacement is generated from the original training image in that exact slot, with the recorded rotation and brightness plus the same crop and JPEG settings. This preserves the training photo identities while matching the number of transformed slots. The comparison with test-derived replacements measures the combined effect of replacing training identities and exposing test identities. Interpret it alongside the comparison with the 0% control.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4
import hashlib
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageEnhance
from sklearn.metrics import f1_score
from google.colab import drive

drive.mount('/content/drive')
project = Path('/content/drive/MyDrive/YOUR_ROCOLE_PROJECT')
split_file = project / 'results/splits/combined_group_split.csv'
root = project / 'results/controlled_leakage_injection'
campaign = root / 'YOUR_REPEAT_CAMPAIGN'
runs = {42: root / 'YOUR_SEED_42_RUN'}
for seed in (137, 911):
    candidates = list(campaign.glob(f'seed_{seed}_*'))
    assert len(candidates) == 1, f'Expected exactly one saved run for seed {seed}: {candidates}'
    runs[seed] = candidates[0]

split_hash = hashlib.sha256(split_file.read_bytes()).hexdigest()
split = pd.read_csv(split_file)
train, val, test = [split.loc[split.split.eq(part)].reset_index(drop=True)
                    for part in ('train', 'val', 'test')]
assert tuple(map(len, (train, val, test))) == (970, 208, 211)
assert split.image_path.is_unique
assert tf.config.list_physical_devices('GPU'), 'Select a GPU in Colab before running.'
plans = {}
for seed, path in runs.items():
    protocol = json.loads((path / 'protocol.json').read_text())
    assert protocol['seed'] == seed and protocol['split_sha256'] == split_hash
    assert tf.__version__ == protocol['tensorflow'], (tf.__version__, protocol['tensorflow'])
    manifest = pd.read_csv(path / 'dose_40/training_manifest.csv')
    assert manifest.original_train_path.equals(train.image_path)
    assert manifest.binary_class.equals(train.binary_class)
    injected = manifest.loc[manifest.test_source.notna()].copy()
    assert len(injected) == 388 and injected.variant_id.is_unique
    assert injected.groupby('binary_class').size().to_dict() == {'healthy': 220, 'rust': 168}
    assert injected.test_source.map(test.set_index('image_path').binary_class).equals(injected.binary_class)
    assert np.isfinite(injected[['rotation_degrees', 'brightness_factor']]).all().all()
    scores = pd.read_csv(path / 'metrics.csv').set_index('leakage_percent', verify_integrity=True)
    assert {0, 40}.issubset(scores.index)
    for part, frame in [('validation', val), ('test', test)]:
        saved = pd.read_csv(path / f'{part}_manifest.csv')
        assert saved.image_path.equals(frame.image_path)
        assert saved.binary_class.equals(frame.binary_class)
    plans[seed] = (injected, scores)
print('Verified all three source runs, split fingerprints, and replacement manifests.')


def decode(path, label):
    image = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, (224, 224))
    return tf.keras.applications.mobilenet_v2.preprocess_input(image), tf.cast(label, tf.float32)

def dataset(frame, seed=None):
    paths = [str(p) for p in frame.image_path]
    labels = frame.binary_class.eq('rust').astype(int).tolist()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if seed is not None:
        ds = ds.shuffle(len(frame), seed=seed, reshuffle_each_iteration=True)
    return ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

def build_model(seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    base = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.layers.GlobalAveragePooling2D()(base(inputs, training=False))
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    model = tf.keras.Model(inputs, tf.keras.layers.Dense(1, activation='sigmoid')(x))
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model



In [ ]:
def evaluate(model, test_ds, truth):
    prob = model.predict(test_ds, verbose=0).ravel()
    assert len(prob) == len(truth) and np.isfinite(prob).all()
    pred = (prob >= 0.5).astype(int)
    conf, correct = np.maximum(prob, 1 - prob), pred == truth
    ece = 0.0
    edges = np.linspace(0, 1, 11)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    metrics = dict(accuracy=correct.mean(), macro_f1=f1_score(truth, pred, average='macro'), ece=ece)
    return metrics, prob, pred

def run_control(seed, injected, scores, folder):
    folder.mkdir()
    model = build_model(seed)
    digest = hashlib.sha256(b''.join(w.tobytes() for w in model.get_weights())).hexdigest()
    assert digest == scores.loc[40, 'initial_weights_sha256'] == scores.loc[0, 'initial_weights_sha256']
    print(f'Seed {seed}: verified initial weights. Creating training-derived variants.', flush=True)
    frame = train.copy()
    truth = test.binary_class.eq('rust').to_numpy(dtype=int)
    val_ds, test_ds = dataset(val), dataset(test)
    audit = injected.assign(training_slot=injected.index, training_donor=injected.original_train_path)
    audit.to_csv(folder / 'matched_replacements.csv', index=False)
    checkpoint = folder / 'best.keras'
    with TemporaryDirectory() as tmp:
        for slot, record in injected.iterrows():
            target = Path(tmp) / f'{slot:04}.jpg'
            with Image.open(record.original_train_path) as original:
                image = original.convert('RGB').rotate(float(record.rotation_degrees), resample=Image.Resampling.BILINEAR)
                w, h = image.size
                dx, dy = round(w * 0.04), round(h * 0.04)
                image = image.crop((dx, dy, w - dx, h - dy))
                ImageEnhance.Brightness(image).enhance(float(record.brightness_factor)).save(target, quality=92)
            frame.at[slot, 'image_path'] = str(target)
        history = model.fit(dataset(frame, seed), validation_data=val_ds, epochs=10, verbose=2,
            callbacks=[tf.keras.callbacks.ModelCheckpoint(str(checkpoint), monitor='val_loss', mode='min', save_best_only=True),
                       tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])
    pd.DataFrame(history.history).to_csv(folder / 'history.csv', index=False)
    metrics, prob, pred = evaluate(tf.keras.models.load_model(checkpoint), test_ds, truth)
    test.assign(y_true=truth, y_pred=pred, rust_probability=prob).to_csv(folder / 'predictions.csv', index=False)
    pd.DataFrame([dict(seed=seed, initial_weights_sha256=digest, **metrics)]).to_csv(folder / 'metrics.csv', index=False)
    return dict(seed=seed, **{
        f'{metric}_{label}_pp': 100 * difference
        for metric, value in metrics.items()
        for label, difference in (
            ('test_derived_minus_training_derived', scores.loc[40, metric] - value),
            ('training_derived_minus_clean', value - scores.loc[0, metric]),
        )
    })

out = project / 'results/matched_augmentation_control' / uuid4().hex[:12]
out.mkdir(parents=True, exist_ok=False)
(out / 'protocol.json').write_text(json.dumps(dict(split_sha256=split_hash, tensorflow=tf.__version__,
    source_runs={seed: str(path) for seed, path in runs.items()}, training_donor_rule='Original image at the same slot'), indent=2))
print('Matched control folder:', out, flush=True)
rows = []
for seed, (injected, scores) in plans.items():
    rows.append(run_control(seed, injected, scores, out / f'seed_{seed}'))
    pd.DataFrame(rows).to_csv(out / 'paired_changes_by_seed.csv', index=False)
summary = pd.DataFrame(rows).drop(columns='seed').agg(['mean', 'std', 'count']).T
summary.to_csv(out / 'paired_summary.csv')
display(summary.round(3))
print('Completed three-seed matched augmentation control:', out)
